> **说明**：本 notebook 是 [langchain-ai/langgraph-101](https://github.com/langchain-ai/langgraph-101) 的**简体中文翻译版**，供 Google Colab / Jupyter 使用。
>
> - 原文仓库：<https://github.com/langchain-ai/langgraph-101>
> - 建议学习顺序：`101_langchain_langgraph` → `102_middleware` → `deep_agents` → `deep_agents_extra`
> - **代码单元格保持英文原样**（标识符、字符串、prompt 均未改动）；仅翻译 Markdown 与 `#` 注释。
> - 在 Colab 中请先运行 **Colab 一键环境** 单元格；图片与 `utils/` 已包含在本仓库中。


# Deep Agents：从零构建研究智能体

欢迎来到 Deep Agents！本 notebook 将通过**循序渐进地从零构建研究智能体**，带你掌握 Deep Agents 框架的核心概念。

**你将学到：**
- Deep Agents 是什么，以及开箱即用提供什么
- 如何添加自定义工具以扩展智能体能力
- 理解后端（backends）与存储抽象
- 用子智能体（subagents）委派任务与隔离上下文
- 面向安全的人机协同（HITL）模式
- 用于持久存储的长期记忆
- 中间件架构与可扩展性
- 可复用的智能体技能（Skills）

<br>
<br>
---
<br>

> **说明：** 本实验需要 [Tavily API key](https://tavily.com) 以使用网页搜索。Deep Agents 构建于 LangGraph 之上，为自主智能体提供文件系统访问、规划与委派等能力。


## 第 0 部分：环境准备与安装

首先安装必要依赖并配置环境。


In [ ]:
# Colab 一键环境：安装依赖、在 Colab 中克隆本仓库（若缺少 utils），并配置 sys.path
# 本地完整 clone 时可跳过 pip/clone，但仍建议运行本单元格以确保路径正确
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/Sucran/langgraph-101-zh.git"
COLAB_REPO_DIR = Path("/content/langgraph-101-zh")


def _in_colab() -> bool:
    if Path("/content").exists():
        return True
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False


if _in_colab():
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "--pre",
            "-U",
            "langchain",
            "langchain-anthropic",
            "langchain-openai",
            "python-dotenv",
            "deepagents",
            "tavily-python",
        ]
    )
    if not (COLAB_REPO_DIR / "utils" / "models.py").is_file():
        subprocess.check_call(
            ["git", "clone", "--depth", "1", REPO_URL, str(COLAB_REPO_DIR)]
        )


def find_project_root() -> Path:
    """Repo root containing utils/models.py (Colab clone preferred when present)."""
    if COLAB_REPO_DIR.is_dir() and (COLAB_REPO_DIR / "utils" / "models.py").is_file():
        return COLAB_REPO_DIR
    for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if (p / "utils" / "models.py").is_file():
            return p
    return Path.cwd().resolve().parent.parent


project_root = find_project_root()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

if not os.environ.get("ANTHROPIC_API_KEY") and not os.environ.get("OPENAI_API_KEY"):
    print(
        "请设置 API Key：Colab 左侧「密钥」→ ANTHROPIC_API_KEY（默认模型）或 OPENAI_API_KEY；"
        "也可执行：import os; os.environ['ANTHROPIC_API_KEY'] = '...'"
    )
if not os.environ.get("TAVILY_API_KEY"):
    print(
        "Deep Agents 需要 TAVILY_API_KEY：Colab「密钥」或 os.environ['TAVILY_API_KEY'] = '...'"
    )


### 初始化你的 LLM


In [ ]:
# 将项目根目录加入 Python 路径，以便从 utils 模块导入
# 若跳过「Colab 一键环境」，本单元格会在 Colab 中自动克隆本仓库以定位 utils/
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/Sucran/langgraph-101-zh.git"
COLAB_REPO_DIR = Path("/content/langgraph-101-zh")


def _in_colab() -> bool:
    if Path("/content").exists():
        return True
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False


def _utils_available() -> bool:
    if (COLAB_REPO_DIR / "utils" / "models.py").is_file():
        return True
    for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if (p / "utils" / "models.py").is_file():
            return True
    return False


def _ensure_repo_for_utils() -> None:
    if _utils_available():
        return
    if _in_colab():
        subprocess.check_call(
            ["git", "clone", "--depth", "1", REPO_URL, str(COLAB_REPO_DIR)]
        )


def find_project_root() -> Path:
    """Repo root containing utils/models.py (Colab clone preferred when present)."""
    if COLAB_REPO_DIR.is_dir() and (COLAB_REPO_DIR / "utils" / "models.py").is_file():
        return COLAB_REPO_DIR
    for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if (p / "utils" / "models.py").is_file():
            return p
    return Path.cwd().resolve().parent.parent

_ensure_repo_for_utils()
project_root = find_project_root()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# 从集中式 utils 模块导入 model
from utils.models import model

# 加载 Tavily 的环境变量
from dotenv import load_dotenv
load_dotenv(dotenv_path=project_root / ".env", override=True)

# 抑制警告以获得更干净的输出
import warnings
warnings.filterwarnings('ignore', message='LangSmith now uses UUID v7')


## 第 1 部分：你的第一个 Deep Agent（智能体「驾驭层」Harness）

<img src="../../images/deepAgentsDiag.png" style="width: auto; height: 500px; border-radius: 8px;" alt="Deep Agents Architecture">

Deep Agents 充当**「智能体驾驭层（agent harness）」**—— 基于核心工具调用循环构建的框架，同时内置预构建工具与自主任务执行能力。

### 开箱即用你得到：

- **文件系统工具** —— `ls`、`read_file`、`write_file`、`edit_file`、`glob`、`grep`
- **规划工具** —— 用于任务跟踪的 `write_todos`
- **子智能体委派** —— 用于隔离工作的 `task()` 工具
- **大工具结果卸载（Eviction）** —— 自动把超过约 20k tokens 的工具结果卸载到文件系统
- **对话摘要（Summarization）** —— 接近约 85% 上下文容量时压缩历史
- **悬空工具调用修补** —— 自动修复消息历史一致性

创建最基础的 deep agent：


In [ ]:
from deepagents import create_deep_agent

# 创建最基础的 deep agent —— 只要一个模型！
agent = create_deep_agent(
    model=model,
    system_prompt="You are a helpful research assistant. When referencing file paths, use backtick formatting like `path/file.md` instead of markdown links."
)

print("Deep agent created successfully!")

### 测试内置文件系统工具

即使不添加任何自定义工具，智能体也已具备文件系统能力：


In [ ]:
# 让智能体写入并读取一个文件
result = agent.invoke({
    "messages": [{"role": "user", "content": "Write a file called notes.md with the text 'Hello from Deep Agents!' then read it back to confirm."}]
})

# 打印最终回复
print("Agent response:", result["messages"][-1].content)

In [ ]:
# 显示虚拟文件系统内容（存在 result["files"] 中）
print("\n" + "=" * 50)
print("📁 VIRTUAL FILESYSTEM (in-memory, not on disk!)")
print("=" * 50)
for path, file_data in result.get("files", {}).items():
    print(f"\n  Path: '{path}'")
    print("  " + "-" * 38)
    # file_data 包含 'content'（行列表）、'created_at'、'modified_at'
    if isinstance(file_data, dict) and "content" in file_data:
        content = "\n".join(file_data["content"])
        for line in content.split("\n"):
            print(f"  | {line}")
    else:
        print(f"  | {file_data}")

### 要点：
- `create_deep_agent()` 免费提供文件系统 + 规划能力
- 无需自己定义基础文件操作 —— 它们是内置的
- 文件存储在智能体状态中（第 3 部分会更深入）


## 第 2 部分：添加自定义工具

内置工具很强大，但构建研究智能体还需要**自定义工具**。下面添加网页搜索与策略性思考能力。

### 用 Tavily 创建网页搜索工具


In [ ]:
from langchain_core.tools import tool
from tavily import TavilyClient

# 初始化 Tavily 客户端
tavily_client = TavilyClient()


@tool(parse_docstring=True)
def tavily_search(query: str) -> str:
    """Search the web for information on a given query.
    
    Args:
        query: Search query to execute
    """
    search_results = tavily_client.search(query, max_results=3, topic="general")
    
    result_texts = []
    for result in search_results.get("results", []):
        url = result["url"]
        title = result["title"]
        content = result.get("content", "No content available")
        result_text = f"## {title}\n**URL:** {url}\n\n{content}\n\n---\n"
        result_texts.append(result_text)
    
    return f"Found {len(result_texts)} result(s) for '{query}':\n\n{''.join(result_texts)}"


print("tavily_search tool created!")

### 把自定义工具加入智能体


In [ ]:
# 用我们的自定义工具重新创建智能体
agent = create_deep_agent(
    model=model,
    tools=[tavily_search],  # Add our custom tools
    system_prompt="""You are a helpful research assistant.
    
Use tavily_search to find information on the web.
When referencing file paths, use backtick formatting like `path/file.md` instead of markdown links.
"""
)

print("Agent updated with custom tools!")

In [ ]:
# 测试搜索能力
result = agent.invoke({
    "messages": [{"role": "user", "content": "Search for information about LangGraph and summarize what you find."}]
})

print(result["messages"][-1].content)

### 要点：
- 自定义工具扩展智能体能做的事
- `@tool` 装饰器把函数转换成 LangChain 工具


## 第 3 部分：理解后端（Backends）

<img src="../../images/deepAgentBackends.png" style="width: auto; height: 500px; border-radius: 8px;" alt="Backend Architecture">

智能体的文件实际写到哪里？**后端（Backends）** 是可插拔的存储系统，向智能体暴露文件系统表面。

### 四种后端类型：

| 后端 | 存储位置 | 持久性 | 适用场景 |
|---------|---------|-------------|----------|
| **StateBackend** | 内存（智能体状态） | 单线程 | 草稿本、中间结果 |
| **FilesystemBackend** | 本地磁盘 | 永久 | 直接文件访问（慎用！） |
| **StoreBackend** | LangGraph Store | 跨线程 | 长期记忆 |
| **CompositeBackend** | 路由到其他后端 | 混合 | 选择性持久化 |

默认情况下，`create_deep_agent()` 使用 **StateBackend** —— 文件存在智能体状态里，线程结束即消失。


In [ ]:
from langgraph.checkpoint.memory import MemorySaver
from langsmith import uuid7

# 添加检查点，以便演示跨轮次持久化
checkpointer = MemorySaver()

agent = create_deep_agent(
    model=model,
    tools=[tavily_search],
    system_prompt="You are a helpful research assistant. When referencing file paths, use backtick formatting like `path/file.md` instead of markdown links.",
    checkpointer=checkpointer
)

# 创建线程
thread_id = str(uuid7())
config = {"configurable": {"thread_id": thread_id}}

print(f"Thread ID: {thread_id}")

In [ ]:
# 在此线程中写入文件
result = agent.invoke({
    "messages": [{"role": "user", "content": "Write a file called /research_notes.md with 'My research findings go here'"}]
}, config=config)

print("Files in state:", list(result.get("files", {}).keys()))

In [ ]:
# 在同一线程中，文件会保留
result = agent.invoke({
    "messages": [{"role": "user", "content": "Read the file `/research_notes.md`"}]
}, config=config)

print(result["messages"][-1].content)

In [ ]:
# 在新线程中，文件已消失（StateBackend 是短暂的）
new_config = {"configurable": {"thread_id": str(uuid7())}}

result = agent.invoke({
    "messages": [{"role": "user", "content": "List all files with ls /"}]
}, config=new_config)

print(result["messages"][-1].content)

### FilesystemBackend —— 写入真实磁盘

当需要智能体操作**磁盘上的真实文件**时，使用 `FilesystemBackend`。开启 `virtual_mode=True` 时，路径会被沙箱限制在 `root_dir` 下，更安全。

> ⚠️ **注意**：FilesystemBackend 会写真实文件！请使用 `virtual_mode=True` 防止路径穿越攻击。


In [ ]:
from deepagents.backends import FilesystemBackend
import tempfile
import os

# 为沙箱创建临时目录
sandbox_dir = tempfile.mkdtemp(prefix="deepagents_sandbox_")
print(f"Sandbox directory: `{sandbox_dir}`")

# 创建 virtual_mode=True 的 FilesystemBackend（沙箱化）
fs_backend = FilesystemBackend(root_dir=sandbox_dir, virtual_mode=True)

# 创建写入真实磁盘的智能体（沙箱化）
agent_with_fs = create_deep_agent(
    model=model,
    system_prompt="You are a helpful assistant. Files you write go to the local filesystem. When referencing file paths, use backtick formatting like `path/file.md` instead of markdown links.",
    backend=fs_backend,
    checkpointer=checkpointer
)

print("Agent with FilesystemBackend created!")

In [ ]:
# 测试：通过智能体写入文件
config = {"configurable": {"thread_id": str(uuid7())}}

result = agent_with_fs.invoke({
    "messages": [{"role": "user", "content": "Write a file called notes.txt with 'Hello from FilesystemBackend!'"}]
}, config=config)

print("Agent response:", result["messages"][-1].content)

# 验证文件确实已写入磁盘！
actual_path = os.path.join(sandbox_dir, "notes.txt")
if os.path.exists(actual_path):
    with open(actual_path, "r") as f:
        print(f"\n✅ File exists on disk at: `{actual_path}`")
        print(f"   Content: {f.read()}")
else:
    print(f"\n❌ File not found at: {actual_path}")

# 列出沙箱中的所有文件
print(f"\n📁 Files in sandbox ({sandbox_dir}):")
for f in os.listdir(sandbox_dir):
    print(f"   - {f}")

### CompositeBackend —— 把路径路由到不同后端

`CompositeBackend` 可将不同路径路由到不同后端。这就是实现**混合存储**的方式 —— 有的路径短暂，有的持久，有的落盘。

```
/                 → StateBackend（短暂草稿空间）
/memories/*       → StoreBackend（跨线程持久）
/workspace/*      → FilesystemBackend（真实磁盘）
```


In [ ]:
from deepagents.backends import StateBackend, CompositeBackend

# 为 workspace 文件创建另一个沙箱
workspace_dir = tempfile.mkdtemp(prefix="deepagents_workspace_")
print(f"Workspace directory: `{workspace_dir}`")

composite_backend = CompositeBackend(
    default=StateBackend(),
    routes={
        # /workspace/* → FilesystemBackend（真实磁盘，沙箱化）
        "/workspace/": FilesystemBackend(root_dir=workspace_dir, virtual_mode=True),
    }
)

agent_composite = create_deep_agent(
    model=model,
    system_prompt="""You are a helpful assistant.

STORAGE RULES:
- Files in /workspace/* are saved to real disk (persistent)
- All other files are ephemeral (disappear when thread ends)

When referencing file paths, use backtick formatting like `path/file.md` instead of markdown links.
""",
    backend=composite_backend,
    checkpointer=checkpointer
)

print("Agent with CompositeBackend created!")

In [ ]:
# 测试：写入两个位置并观察差异
config = {"configurable": {"thread_id": str(uuid7())}}

result = agent_composite.invoke({
    "messages": [{"role": "user", "content": """Write two files:
1. `/workspace/persistent.txt` with 'I will survive!'
2. `/scratch.txt` with 'I am ephemeral'

Then list both locations."""}]
}, config=config)

print("Agent response:", result["messages"][-1].content)

# 检查磁盘上有什么 vs 状态中有什么
print("\n" + "=" * 50)
print("📁 ON DISK (workspace_dir):")
for f in os.listdir(workspace_dir):
    filepath = os.path.join(workspace_dir, f)
    with open(filepath, "r") as file:
        print(f"   {f}: {file.read()}")

print("\n📦 IN VIRTUAL STATE:")
for path, data in result.get("files", {}).items():
    if isinstance(data, dict) and "content" in data:
        content = "\n".join(data["content"])
    else:
        content = str(data)
    print(f"   `{path}`: {content}")

### StoreBackend —— 用于 LangSmith 部署

`StoreBackend` 使用 LangGraph 的 `BaseStore` 做持久、跨线程存储。你会在以下场景使用它：

- 本地运行 `langgraph dev`（平台会自动提供 store）
- 部署到 **LangSmith Deployments**（由平台托管 store）

平台提供持久 store，因此智能体的 `/memories/*` 文件可跨会话与服务器重启存活。

> 💡 **说明**：我们将在第 6 部分（长期记忆）演示 `StoreBackend`，并与 `CompositeBackend` 组合做选择性持久化。

```python
# When deployed to LangSmith Deployment, the store is injected automatically
# This is the pattern you'll use in production:
from deepagents.backends import StateBackend, StoreBackend, CompositeBackend
from langgraph.store.memory import InMemoryStore

agent = create_deep_agent(
    backend=CompositeBackend(
        default=StateBackend(),  # Everything else is ephemeral
        routes={
            # /memories/* persists across threads via the platform's store
            "/memories/": StoreBackend(),
        }
    ),
    store=InMemoryStore()  # Good for local dev; omit for LangSmith Deployment
)
```


In [ ]:
# 清理：删除临时目录
import shutil
shutil.rmtree(sandbox_dir, ignore_errors=True)
shutil.rmtree(workspace_dir, ignore_errors=True)
print("✅ Temporary directories cleaned up")

### 要点：
- **后端（Backends）** 控制智能体文件存哪里、怎么存
- **StateBackend**（默认）是短暂的 —— 线程结束文件即消失
- **FilesystemBackend** 写入真实磁盘（用 `virtual_mode=True` 做沙箱）
- **CompositeBackend** 把不同路径路由到不同后端（混合存储）
- **StoreBackend** 用于 LangSmith Deployment（第 6 部分会用到）




## 第 4 部分：添加研究子智能体（Subagent）

<img src="../../images/deepAgentSubagents.png" style="width: auto; height: 500px; border-radius: 8px;" alt="Subagent Architecture">

智能体做的事越多，上下文越容易被中间工具调用填满。**子智能体（Subagents）** 通过在独立上下文中隔离工作来解决这个问题。

### 上下文膨胀问题

没有子智能体时：
```
User: Research AI agents
→ search("AI agents overview") → 5000 tokens of results
→ think("Found overview...") → 200 tokens
→ search("AI agent frameworks") → 5000 tokens of results  
→ think("Found frameworks...") → 200 tokens
→ ... context bloats with every search
```

有子智能体时：
```
User: Research AI agents
→ task("research-agent", "Research AI agents")
→ [Subagent does all searches in isolated context]
→ Returns: "Summary: AI agents are..." (clean, compressed result)
```



In [ ]:
from datetime import datetime

# 获取研究者所需的当前日期
current_date = datetime.now().strftime("%Y-%m-%d")

# 定义研究者子智能体
RESEARCHER_INSTRUCTIONS = f"""You are a research assistant conducting research. Today's date is {current_date}.

<Task>
Use tools to gather information about the research topic.
</Task>

<Hard Limits>
- Simple queries: Use 2-3 search tool calls maximum
- Complex queries: Use up to 5 search tool calls maximum
</Hard Limits>

<Output Format>
Structure your findings with:
- Clear headings
- Inline citations [1], [2], [3]
- Sources section at the end
</Output Format>

When referencing file paths, use backtick formatting like `path/file.md` instead of markdown links.
"""

research_subagent = {
    "name": "research-agent",
    "description": "Delegate research tasks. Give one topic at a time.",
    "system_prompt": RESEARCHER_INSTRUCTIONS,
    "tools": [tavily_search],
}

print("Research subagent defined!")
print(f"  Name: {research_subagent['name']}")
print(f"  Tools: {[t.name for t in research_subagent['tools']]}")

In [ ]:
# 定义编排器指令
ORCHESTRATOR_INSTRUCTIONS = """You are a research coordinator.

When asked to research a topic:
1. Use write_todos to plan your research tasks
2. Delegate research to the research-agent subagent using the task() tool
3. NEVER search directly - always delegate to the research-agent
4. Synthesize findings and write a report to /final_report.md

The research-agent will handle all web searches and return summarized findings.

When referencing file paths, use backtick formatting like `path/file.md` instead of markdown links.
"""

# 创建带子智能体的智能体
agent = create_deep_agent(
    model=model,
    tools=[tavily_search],
    system_prompt=ORCHESTRATOR_INSTRUCTIONS,
    subagents=[research_subagent],  # Add our research subagent
    checkpointer=checkpointer
)

print("Agent with subagent created!")

In [ ]:
# 测试委派
config = {"configurable": {"thread_id": str(uuid7())}}

result = agent.invoke({
    "messages": [{"role": "user", "content": "Lightly research this week's intersesting news on AI agents"}]
}, config=config)

print(result["messages"][-1].content[:2000] + "...")

### 要点：
- 子智能体在独立上下文中隔离工作
- 主智能体只看到最终结果，看不到中间搜索
- 这能让主智能体上下文保持干净、聚焦


## 第 5 部分：中间件深挖

Deep Agents 使用**模块化中间件架构**。调用 `create_deep_agent()` 时，会自动挂接若干中间件组件。
<img src="../../images/deepAgentMiddleware.png" style="width: auto; height: 500px; border-radius: 8px;" alt="Middleware">

### 始终启用的中间件：

| 中间件 | 提供的工具 | 用途 |
|------------|---------------|----------|
| **TodoListMiddleware** | `write_todos` | 任务规划与跟踪 |
| **FilesystemMiddleware** | `ls`、`read_file`、`write_file`、`edit_file`、`glob`、`grep` | 文件操作 + **大工具结果卸载** |
| **SubAgentMiddleware** | `task` | 向子智能体委派工作 |
| **SummarizationMiddleware** | *（无）* | 在约 85% 上下文容量时压缩对话历史 |
| **PatchToolCallsMiddleware** | *（无）* | 修复消息历史中的悬空工具调用 |

### 条件中间件（配置后才会加入）：

| 中间件 | 触发条件 | 用途 |
|------------|---------|----------|
| **MemoryMiddleware** | `memory=["/AGENTS.md"]` | 从 AGENTS.md 加载持久上下文 |
| **SkillsMiddleware** | `skills=["/skills/"]` | 按需渐进披露打包能力 |
| **HumanInTheLoopMiddleware** | `interrupt_on={...}` | 对敏感操作做人工批准 |

### 内置上下文管理策略

Deep agents 会自动管理上下文以遵守模型的 token 限制。有三种策略，均可经 `tool_token_limit_before_evict` 与模型的 `max_input_tokens` 配置：

<style>
.ctx-grid { display: grid; grid-template-columns: repeat(auto-fit, minmax(500px, 1fr)); gap: 10px; margin: 10px 0 10px 0; }
.ctx-card img { width: 1000px; border-radius: 8px; margin-bottom: 8px; }
.ctx-card p { font-size: 0.92em; line-height: 1.5; margin: 0; }
.ctx-card strong { display: block; margin-bottom: 4px; }
</style>
<div class="ctx-grid">
<div class="ctx-card">
<img src="../../images/Offloading%20Inputs%20LangChain.png" alt="Offloading tool inputs">
<p><strong>卸载大型输入（Offload Large Inputs）</strong>文件写入 / 编辑的工具调用会把完整内容留在对话历史里。既然已在磁盘上，这些内容是冗余的。在约 85% 上下文容量时，deep agents 会截断这些较旧的工具调用，并替换为指向文件的指针。</p>
</div>
<div class="ctx-card">
<img src="../../images/Offloading%20Results%20LangChain.png" alt="Offloading tool results">
<p><strong>卸载大型结果（Offload Large Results）</strong>当工具结果超过约 20k tokens 时，deep agent 会将其保存到后端，并替换为文件路径引用 + 10 行预览。智能体可按需重新读取或搜索全文。</p>
</div>
<div class="ctx-card">
<img src="../../images/LangChain%20Summarization.png" alt="Conversation summarization">
<p><strong>对话摘要（Conversation Summarization）</strong>当上下文达到 <code>max_input_tokens</code> 的约 85%、且已无可卸载内容时，智能体会摘要历史。完整消息会保留到磁盘上的 <code>/conversation_history/</code>；工作记忆中则用结构化摘要替换。</p>
</div>
</div>


In [ ]:
# 我们的智能体已具备全部三类中间件！
# 看看规划能力实际效果

config = {"configurable": {"thread_id": str(uuid7())}}

result = agent.invoke({
    "messages": [{"role": "user", "content": "Create a plan for researching machine learning frameworks. Use write_todos."}]
}, config=config)

print(result["messages"][-1].content)

In [ ]:
# 检查状态中的 todos
if "todos" in result:
    print("📋 Agent's Todo List:\n")
    for todo in result["todos"]:
        status_map = {"completed": "✅", "in_progress": "🔄", "pending": "⬚"}
        status = todo.get("status", "pending")
        icon = status_map.get(status, "⬚")
        content = todo.get("content", str(todo))
        print(f"  {icon} {content}")
else:
    print("No todos in state (agent may have used a different approach)")

### 自定义中间件：记录工具调用

除内置中间件外，你也可以自己写。最简单的方式是用**基于装饰器的中间件**，例如 `@wrap_tool_call`，它会包裹每一次工具调用。

这对可观测性很有用 —— 实时看到智能体在做什么。

可用 hooks：

| Hook | 何时运行 | 风格 |
|------|-------------|-------|
| `@before_agent` | 智能体开始前（一次） | 节点 |
| `@before_model` | 每次 LLM 调用前 | 节点 |
| `@after_model` | 每次 LLM 响应后 | 节点 |
| `@after_agent` | 智能体完成后（一次） | 节点 |
| `@wrap_model_call` | 包裹每次 LLM 调用 | 包裹 |
| `@wrap_tool_call` | 包裹每次工具调用 | 包裹 |

创建一个简单的 `@wrap_tool_call` 中间件，记录智能体使用的每个工具：


In [ ]:
from langchain.agents.middleware import wrap_tool_call

@wrap_tool_call
def log_tool_calls(request, handler):
    """Log every tool call the agent makes."""
    tool_name = request.tool_call["name"]
    tool_args = request.tool_call["args"]
    print(f"🔧 [Tool Call] {tool_name}")
    print(f"   Args: {tool_args}")

    result = handler(request)

    print(f"✅ [Tool Done] {tool_name}\n")
    return result


# 创建带我们自定义中间件的智能体
agent_with_logging = create_deep_agent(
    model=model,
    tools=[tavily_search],
    system_prompt="You are a helpful research assistant. When referencing file paths, use backtick formatting like `path/file.md` instead of markdown links.",
    middleware=[log_tool_calls],
    checkpointer=MemorySaver(),
)

In [ ]:
# 运行智能体 —— 在输出中观察工具调用日志！
config = {"configurable": {"thread_id": str(uuid7())}}

result = agent_with_logging.invoke({
    "messages": [{"role": "user", "content": "What is LangGraph? create a short summary in your filesystem."}]
}, config=config)

print("\n--- Agent Response ---")
print(result["messages"][-1].content)

### 要点：
- 中间件 = 可插拔能力模块，包裹智能体的模型调用与工具调用
- `create_deep_agent()` 会自动加入 TodoList、Filesystem、SubAgent、Summarization 与 PatchToolCalls 中间件
- **上下文管理是内置的**：大工具结果会被卸载，对话历史会被摘要
- 写自定义中间件很直接 —— 用 `@wrap_tool_call` 等装饰器快速做单用途中间件，或用基于类的 `AgentMiddleware` 做更复杂场景
- 通过 `middleware=[...]` 传入自定义中间件，它会与全部内置中间件组合


## 第 6 部分：人机协同（HITL）

<img src="../../images/deepAgentHITL.png" style="width: auto; height: 500px; border-radius: 8px;" alt="Human-in-the-Loop Flow">

对敏感操作，你可能希望人类在执行前批准。Deep Agents 通过 `HumanInTheLoopMiddleware` 内置支持**中断（interrupts）**。

### 内置决策类型：
- **Approve（批准）** —— 按提议参数执行
- **Edit（编辑）** —— 执行前修改参数
- **Reject（拒绝）** —— 完全跳过该工具调用


你也可以为每个工具自定义可用决策，例如：
```python
interrupt_on = {
    # Sensitive operations: allow all options
    "delete_file": {"allowed_decisions": ["approve", "edit", "reject"]},

    # Moderate risk: approval or rejection only
    "write_file": {"allowed_decisions": ["approve", "reject"]},

    # Must approve (no rejection allowed)
    "critical_operation": {"allowed_decisions": ["approve"]},
}
```

#### 本演示使用默认的中断决策类型：


In [ ]:
# 创建在文件写入时中断的智能体
agent_with_hitl = create_deep_agent(
    model=model,
    tools=[tavily_search],
    system_prompt="You are a helpful research assistant. When referencing file paths, use backtick formatting like `path/file.md` instead of markdown links.",
    subagents=[research_subagent],
    checkpointer=checkpointer,
    interrupt_on={
        "write_file": True,  # Interrupt before writing files
        "edit_file": True,   # Interrupt before editing files
    }
)

print("Agent with HITL created!")

#### 试一试！


In [ ]:
from langgraph.types import Command

config = {"configurable": {"thread_id": str(uuid7())}}

# 智能体尝试写文件时会触发中断
result = agent_with_hitl.invoke({
    "messages": [{"role": "user", "content": "Write a file called /test.md with 'Hello World'"}]
}, config=config)

# 检查是否命中了中断
if result.get("__interrupt__"):
    print("🛑 Interrupt triggered!\n")
    interrupt_value = result["__interrupt__"][0].value
    action_requests = interrupt_value["action_requests"]
    review_configs = interrupt_value["review_configs"]

    for action, review in zip(action_requests, review_configs):
        print(f"  Tool: {action['name']}")
        print(f"  Args: {action['args']}")
        print(f"  Allowed decisions: {review['allowed_decisions']}")
else:
    print("No interrupt (file was written)")
    print(result["messages"][-1].content)

#### 批准后恢复


In [ ]:
if result.get("__interrupt__"):
    # 批准写入操作
    result = agent_with_hitl.invoke(
        Command(resume={"decisions": [{"type": "approve"}]}),
        config=config
    )
    print("✅ Resumed with approval!")
    print(result["messages"][-1].content)

### 要点：
- HITL 为高风险操作加入人工监督
- 用 `interrupt_on` 配置哪些工具需要批准
- HITL 需要检查点（checkpointer）才能工作


## 第 7 部分：长期记忆

<img src="../../images/deepAgentMemories.png" style="width: auto; height:500px; border-radius: 8px;" alt="Long-Term Memory">

到目前为止，线程结束文件就会消失。**长期记忆**使用 `CompositeBackend` 把某些路径路由到持久存储。

### 路径路由：
- `/memories/*` → **StoreBackend**（跨线程持久）
- 其他一切 → **StateBackend**（短暂）



In [ ]:
from langgraph.store.memory import InMemoryStore
from deepagents.backends import StateBackend, StoreBackend, CompositeBackend

store = InMemoryStore()

composite_backend = CompositeBackend(
    default=StateBackend(),
    routes={
        # /memories/ 下的文件进入持久 store
        "/memories/": StoreBackend(),
    },
)

print("Composite backend created!")

In [ ]:
# 创建带长期记忆的智能体
agent_with_memory = create_deep_agent(
    model=model,
    tools=[tavily_search],
    system_prompt="""You are a helpful research assistant with long-term memory.
    
IMPORTANT: Save important notes to /memories/ so they persist across conversations.
For example: /memories/research_notes.md

Regular files (not in /memories/) will disappear when the conversation ends.

When asked what you remember or know, ALWAYS use ls and read_file to check /memories/ first.
Do not rely on conversation context alone -- memories persist across threads but conversation does not.

When referencing file paths, use backtick formatting like `path/file.md` instead of markdown links.
""",
    subagents=[research_subagent],
    checkpointer=checkpointer,
    backend=composite_backend,
    store=store,  # Store is passed here, not to the backend directly
)

print("Agent with long-term memory created!")

In [ ]:
# 线程 1：把内容保存到长期记忆
thread1_config = {"configurable": {"thread_id": str(uuid7())}}

result = agent_with_memory.invoke({
    "messages": [{"role": "user", "content": "Save 'Important research finding: AI agents are evolving rapidly' to `/memories/findings.md`"}]
}, config=thread1_config)

print("Thread 1:", result["messages"][-1].content)

In [ ]:
# 线程 2：从另一线程访问记忆
thread2_config = {"configurable": {"thread_id": str(uuid7())}}

result = agent_with_memory.invoke({
    "messages": [{"role": "user", "content": "Read me what's in the file `/memories/findings.md`"}]
}, config=thread2_config)

print("Thread 2:", result["messages"][-1].content)

### 要点：
- `CompositeBackend` 把不同路径路由到不同存储后端
- `/memories/*` 跨线程持久
- 其他文件仍短暂（单线程）


### 进阶记忆模式：语义、情景与程序性

认知科学研究（以及 [CoALA 论文](https://arxiv.org/abs/2309.02427)）识别出三类长期记忆，可自然映射到智能体如何存储信息：

| 记忆类型 | 存储内容 | 人类例子 | 智能体例子 |
|-------------|---------------|---------------|---------------|
| **Semantic（语义）** | 事实与知识 | 「巴黎是法国首都」 | 用户偏好、项目上下文 |
| **Episodic（情景）** | 过往经历 | 「上周二我去徒步了」 | 过往研究会话、交互日志 |
| **Procedural（程序性）** | 指令与规则 | 如何骑自行车 | 编码规范、报告格式规则 |

可用带多条路由的 `CompositeBackend` 把它们映射到**文件系统路径**：

```
/memories/semantic/       -> Facts: user_preferences.md, project_context.md
/memories/episodic/       -> Experiences: 2025-02-17_research.md
/memories/procedural/     -> Rules: coding_standards.md, report_format.md
/                         -> Ephemeral scratch space (StateBackend)
```

由于 CompositeBackend 使用**最长前缀匹配**，`/memories/semantic/` 会优先于更宽的 `/memories/` 路由。


In [ ]:
# 为每种记忆类型创建带路由的 CompositeBackend

memory_store = InMemoryStore()

advanced_composite_backend = CompositeBackend(
    default=StateBackend(),
    routes={
        "/memories/semantic/":   StoreBackend(namespace=lambda rt: ("memories", "semantic")),
        "/memories/episodic/":   StoreBackend(namespace=lambda rt: ("memories", "episodic")),
        "/memories/procedural/": StoreBackend(namespace=lambda rt: ("memories", "procedural")),
    },
)

print("Advanced composite backend created with 3 memory type routes!")

In [ ]:
# 创建理解三类记忆的智能体
memory_agent = create_deep_agent(
    model=model,
    system_prompt="""You are a helpful assistant with structured long-term memory.

Your memory is organized into three types:
- /memories/semantic/   -> Facts & knowledge (user preferences, project details)
- /memories/episodic/   -> Past experiences (session logs, interaction summaries)  
- /memories/procedural/ -> Instructions & rules (how to format reports, coding standards)

When asked to remember something, save it to the appropriate memory type.
Regular files (not in /memories/) are ephemeral and disappear after the conversation.

When asked what you remember or know about the user, ALWAYS use ls and read_file to check
the memory directories first. Do not rely on conversation context alone.

When referencing file paths, use backtick formatting like `path/file.md` instead of markdown links.
""",
    checkpointer=checkpointer,
    backend=advanced_composite_backend,
    store=memory_store,
)

# 线程 1：写入全部三类记忆
config_t1 = {"configurable": {"thread_id": str(uuid7())}}

result = memory_agent.invoke({
    "messages": [{"role": "user", "content": """Please save the following to the appropriate memory types:
1. I prefer Python over JavaScript (this is a fact about me)
2. In our last session, we researched LangGraph and found it useful (this is a past experience)
3. Always use inline citations [1], [2] in research reports (this is a rule)"""}]
}, config=config_t1)

print(result["messages"][-1].content)

In [ ]:
# 线程 2：验证记忆跨线程持久
config_t2 = {"configurable": {"thread_id": str(uuid7())}}

result = memory_agent.invoke({
    "messages": [{"role": "user", "content": "What do you remember about me? Check all memory types: semantic, episodic, and procedural."}]
}, config=config_t2)

print("From a NEW thread:\n")
print(result["messages"][-1].content)

### 命名空间作用域：按用户 vs 全局记忆

默认情况下，`StoreBackend` 把文件存在命名空间 `(assistant_id, "filesystem")` —— 即同一助手的所有用户共享记忆。若想要**按用户隔离**呢？

`StoreBackend` 接受 `namespace` 参数 —— 一个接收 `BackendContext` 并返回命名空间元组的可调用对象。它控制数据隔离：

| 作用域 | 命名空间 | 谁能看到 |
|-------|-----------|----------------|
| **默认** | `(assistant_id, "filesystem")` | 同一助手的所有用户 |
| **按用户** | `("user", user_id, "filesystem")` | 仅该特定用户 |
| **全局** | `("shared", "filesystem")` | 跨所有助手的所有用户 |

这样你可以构建：
- `/memories/user/` 存私有按用户偏好（按 `user_id` 隔离）
- `/memories/shared/` 存团队指南，对所有人可见


In [ ]:
scoped_store = InMemoryStore()

scoped_backend = CompositeBackend(
    default=StateBackend(),
    routes={
        # 按用户记忆：namespace lambda 接收 runtime，
        # 因此 rt.context 能拿到当前调用的 user_id
        "/memories/user/": StoreBackend(
            namespace=lambda rt: (
                "user",
                getattr(rt.context, "user_id", "default"),
                "filesystem",
            ),
        ),
        # 共享记忆：所有用户使用同一命名空间
        "/memories/shared/": StoreBackend(
            namespace=lambda rt: ("shared", "filesystem"),
        ),
    },
)



scoped_agent = create_deep_agent(
    model=model,
    system_prompt="""You are a helpful assistant with scoped memory.

MEMORY SCOPES:
- /memories/user/    -> Private to the current user (only they can see it)
- /memories/shared/  -> Shared across all users (everyone can see it)

Save personal preferences to /memories/user/ and team guidelines to /memories/shared/.

When asked what you remember, ALWAYS use ls and read_file to check memory directories first.

When referencing file paths, use backtick formatting like `path/file.md` instead of markdown links.
""",
    checkpointer=checkpointer,
    backend=scoped_backend,
    store=scoped_store,
)

print("Scoped agent created with per-user and shared memory!")

In [ ]:
# 用户 A 同时写入私有与共享记忆
config_alice = {"configurable": {"thread_id": str(uuid7()), "user_id": "alice"}}

result = scoped_agent.invoke({
    "messages": [{"role": "user", "content": """Save these two things:
1. To my private memory (/memories/user/): 'Alice prefers dark mode and Python'
2. To shared memory (/memories/shared/): 'Team guideline: Always write unit tests'"""}]
}, config=config_alice)

print("Alice wrote:\n", result["messages"][-1].content)

In [ ]:
# 用户 B 尝试读取两者 —— 能否看到 Alice 的私有记忆？
config_bob = {"configurable": {"thread_id": str(uuid7()), "user_id": "bob"}}

result = scoped_agent.invoke({
    "messages": [{"role": "user", "content": "List all files in `/memories/user/` and `/memories/shared/` to see what you can access."}]
}, config=config_bob)

print("Bob sees:\n", result["messages"][-1].content)
print("\n" + "=" * 50)
print("Bob can see shared guidelines but NOT Alice's private preferences!")

### 要点：
- **三类记忆**（语义、情景、程序性）可通过 `CompositeBackend` 路由自然映射到文件系统路径
- **`namespace`** 在 `StoreBackend` 上控制数据隔离 —— 按用户、按助手或全局
- `CompositeBackend` 中更长的路由前缀优先（例如 `/memories/semantic/` 优于 `/memories/`）
- 通过 config 传入 `user_id` 按用户划定记忆：`config={"configurable": {"user_id": "alice"}}`


## 第 8 部分：AGENTS.md 与 Skills

到目前为止我们一直在代码里直接写 `system_prompt` 字符串。Deep Agents 提供两种更强大、更符合最佳实践的基于文件的替代方案：

### AGENTS.md：持久身份与指令

`AGENTS.md` 文件提供通过 `memory=` 参数**始终加载**到系统提示中的持久上下文。适合放智能体身份、工作流规则与偏好。另一大好处是它们是简单 Markdown，任何人都能编辑。

强大之处在于：**智能体可以读取并编辑自己的 AGENTS.md**。这意味着它能根据反馈更新自己的指令 —— 本质上是可自修改的系统提示。

> **权衡**：AGENTS.md 对开发与内部智能体很强 —— 智能体能通过编辑自身指令学习与自改进。但在**生产**中，你应把关键指令放到智能体无法编辑的 `system_prompt`，以防它意外破坏自身行为。可以把 AGENTS.md 看作适合「元学习」阶段。

### Skills：按需能力

Skills 是打包为带 frontmatter 元数据的 `SKILL.md` 文件的可复用能力。它们使用**渐进披露（progressive disclosure）**：
1. 启动时只加载技能的**名称 + 描述**（frontmatter）—— 仅少量 tokens
2. 当智能体判断某技能与当前任务相关时，再读取**完整 SKILL.md** 内容
3. 这样在真正需要前保持提示干净

### 对比

| 方式 | 何时加载 | 智能体可编辑 | 最适合 |
|----------|-------------|-------------------|----------|
| `system_prompt` | 始终 | 否 | 核心身份、不可变规则 |
| `AGENTS.md`（memory） | 始终 | 是 | 工作流、偏好、可学习规则 |
| `SKILL.md`（skills） | 按需 | 否（只读） | 任务特定指令、模板 |



In [ ]:
# 定义我们的 AGENTS.md —— 智能体身份与指令
agents_md_content = """# Research Assistant

You are an expert research assistant that can search the web, synthesize findings and produce polished reports and content.

## Workflow

1. **Plan** -- Use `write_todos` to break the task into steps
2. **Research** -- Search for information using tavily_search
3. **Reflect** -- after each search reflect and analyze findings
4. **Synthesize** -- Combine findings into a comprehensive report
5. **Write** -- Save the final report to `/final_report.md`
6. **Remember** -- Save key takeaways to `/memories/research_notes.md`

## Rules

- Use 2-3 searches maximum
- Consolidate citations -- each unique URL gets one number [1], [2], [3]
- End reports with a Sources section
- Check for relevant skills when asked to create specific content formats
"""

print("AGENTS.md defined!")
print(agents_md_content)

### 接下来定义智能体的 Skills


In [ ]:
# 定义 LinkedIn 帖子技能
linkedin_skill_content = """---
name: linkedin-post
description: Write a LinkedIn post based on research findings or a given topic. Use this skill when asked to create LinkedIn content, professional posts, or thought leadership pieces.
---

# LinkedIn 帖子技能

## Format
- **Hook**: Start with a bold opening line that grabs attention (appears before the 'see more' cut)
- **Body**: 3-5 short paragraphs, each 1-2 sentences
- Use line breaks between paragraphs for readability
- Include 1-2 relevant emojis per paragraph (don't overdo it)
- End with a call-to-action or question to drive engagement
- Add 3-5 relevant hashtags at the bottom

## Tone
- Professional but conversational
- Share insights, not just information
- Use 'I' statements and personal perspective where appropriate

## Length
- Ideal: 150-300 words
- LinkedIn truncates after ~210 characters, so the first line must hook the reader
"""

print("LinkedIn post skill defined!")
print(f"Skill name: linkedin-post")
print(f"Length: {len(linkedin_skill_content)} chars")

In [ ]:
# 定义 Twitter/X 帖子技能
twitter_skill_content = """---
name: twitter-post
description: Write a Twitter/X post or thread based on research findings or a given topic. Use this skill when asked to create tweets, X posts, or social media threads.
---

# Twitter/X 帖子技能

## Single Tweet Format
- Maximum 280 characters
- Lead with the most compelling point
- Use numbers or data when possible
- 1-2 hashtags max (optional)

## Thread Format (for longer content)
- Tweet 1: Hook + preview (e.g., 'A thread on X:')
- Tweets 2-N: One idea per tweet, numbered (1/, 2/, 3/)
- Final tweet: Summary + call-to-action
- 4-8 tweets is the sweet spot

## Tone
- Concise and punchy
- Opinionated takes perform better than neutral summaries
- Use plain language -- no corporate speak
"""

print("Twitter/X post skill defined!")
print(f"Skill name: twitter-post")
print(f"Length: {len(twitter_skill_content)} chars")

### 组合起来（AGENTS.md + Skills）


In [ ]:
# 创建使用 AGENTS.md + skills 的智能体
# 在 notebook 中，通过 `files` 状态键用 create_file_data() 播种文件
# 生产中（langgraph dev / Studio）会用 FilesystemBackend 从磁盘加载
from deepagents.backends.utils import create_file_data

skill_agent = create_deep_agent(
    model=model,
    tools=[tavily_search],
    system_prompt="You are an expert research assistant.",
    memory=["/AGENTS.md"],         # Always loaded into system prompt
    skills=["/skills/"],            # Loaded on demand via progressive disclosure
    checkpointer=checkpointer,
    backend=composite_backend,
    store=store,
)

# 把 AGENTS.md 与 skill 文件播种到虚拟文件系统
skill_files = {
    "/AGENTS.md": create_file_data(agents_md_content),
    "/skills/linkedin-post/SKILL.md": create_file_data(linkedin_skill_content),
    "/skills/twitter-post/SKILL.md": create_file_data(twitter_skill_content),
}

print("Agent created with AGENTS.md + 2 skills!")
print(f"  Memory: `/AGENTS.md` (always loaded)")
print(f"  Skills: linkedin-post, twitter-post (loaded on demand)")

In [ ]:
# 测试：研究一个主题，然后请求 LinkedIn 帖子
# 智能体先研究（无需 skill），再按需加载 linkedin-post skill
config = {"configurable": {"thread_id": str(uuid7())}}

result = skill_agent.invoke(
    {
        "messages": [{"role": "user", "content": "Research what AI agents are, then write a LinkedIn post about your findings."}],
        "files": skill_files
    },
    config=config
)

print(result["messages"][-1].content)

In [ ]:
# 测试：在同一对话中请求推文线程
# twitter-post skill 将按需加载
result = skill_agent.invoke(
    {
        "messages": [{"role": "user", "content": "Now write a Twitter/X thread about the same topic."}]
    },
    config=config
)

print(result["messages"][-1].content)

### 要点：
- **AGENTS.md** 替代硬编码 `system_prompt` 字符串 —— 经 `memory=` 加载，可由智能体编辑
- **Skills** 通过渐进披露提供按需能力 —— 仅在相关时加载
- 在 notebook 中用 `files=` 与 `create_file_data()` 播种文件；生产中用 `FilesystemBackend` 从磁盘加载
- 智能体可通过编辑自己的 AGENTS.md 自改进 —— 开发期很强，但生产中请把关键规则锁定在 `system_prompt`


## 第 9 部分：完整研究智能体

回顾我们构建的内容！从基础的 `create_deep_agent()` 调用开始，我们逐步加入：

```
Part 1: create_deep_agent(model)                    → Basic filesystem agent
Part 2: + tools=[tavily_search]                     → Can search web
Part 3: (understand backends)                       → Same agent, understood storage
Part 4: + subagents=[research_agent]                → Can delegate research
Part 5: + middleware=[log_tool_calls]               → Log tool calls as they happen
Part 6: + interrupt_on={...}, checkpointer          → Human oversight
Part 7: + backend (CompositeBackend)                → Long-term memory
Part 8: + memory (AGENTS.md) + skills (SKILL.md)    → Self-modifiable identity + on-demand capabilities
```

现在用 AGENTS.md + skills 模式构建最终智能体：


In [ ]:
# 创建最终研究智能体 —— 使用 AGENTS.md + skills（与 Studio 智能体相同）
final_agent = create_deep_agent(
    model=model,
    tools=[tavily_search],
    system_prompt="You are an expert research assistant.",
    middleware=[log_tool_calls],
    skills=["/skills/"],            # LinkedIn + Twitter skills loaded on demand
    memory=["/AGENTS.md"],         # Identity + workflow from AGENTS.md
    checkpointer=checkpointer,
    backend=composite_backend,
    store=store,
)

# 把文件播种到虚拟文件系统
final_agent_files = {
    "/AGENTS.md": create_file_data(agents_md_content),
    "/skills/linkedin-post/SKILL.md": create_file_data(linkedin_skill_content),
    "/skills/twitter-post/SKILL.md": create_file_data(twitter_skill_content),
}

print("Final research agent created with AGENTS.md + skills!")

In [ ]:
# 运行完整研究工作流 —— 研究 + LinkedIn 帖子（练习 skill！）
config = {"configurable": {"thread_id": str(uuid7())}}

print("Starting research workflow...\n")

result = final_agent.invoke(
    {
        "messages": [{"role": "user", "content": "Research what LangChain Deep Agents are, write a brief report, and then write a LinkedIn post about your findings."}],
        "files": final_agent_files
    },
    config=config,
)

print(result["messages"][-1].content[:2000])

In [ ]:
# 显示虚拟文件系统 —— 智能体写文件了吗？
print("\n" + "=" * 60)
print("📁 VIRTUAL FILESYSTEM")
print("=" * 60)
for path, file_data in result.get("files", {}).items():
    if isinstance(file_data, dict) and "content" in file_data:
        content = "\n".join(file_data["content"])
    else:
        content = str(file_data)
    print(f"\n📄 '{path}' ({len(content)} chars)")
    print("-" * 40)
    print(content[:500] + ("..." if len(content) > 500 else ""))

### 要点：
- Deep Agents 支持增量构建复杂智能体
- 每项能力（工具、子智能体、HITL、记忆）都可组合
- 框架负责编排，你专注于能力本身


## 第 10 部分：下一步

恭喜！你已用 Deep Agents 构建了完整的研究智能体。你学到了：

| 概念 | 作用 |
|---------|-------------|
| **智能体驾驭层（Agent Harness）** | 预构建工具 + 上下文管理 |
| **自定义工具** | 扩展能力（搜索） |
| **后端（Backends）** | 控制文件存储 —— 短暂、磁盘、持久或混合 |
| **子智能体** | 复杂任务的上下文隔离 |
| **中间件** | 可插拔能力模块（文件系统、摘要、todos 等） |
| **人机协同（HITL）** | 敏感操作的安全门 |
| **长期记忆** | 带路径路由 + 命名空间作用域的持久存储 |
| **AGENTS.md** | 基于文件的智能体身份 —— 始终加载，可由智能体编辑 |
| **Skills** | 通过渐进披露的按需能力（SKILL.md 文件） |

### 资源

- [Deep Agents 文档](https://docs.langchain.com/oss/python/deepagents/)
- [LangGraph 文档](https://docs.langchain.com/oss/python/langgraph/)
- [LangChain Academy](https://academy.langchain.com/)
- [LangChain vs LangGraph vs Deep Agents](https://docs.langchain.com/oss/python/concepts/products)

### 接下来做什么？

1. **在 Studio 中运行** - 查看 `agents/deep_agent/`，这是带磁盘上 AGENTS.md 与 skills 的生产就绪版本
2. **部署** - 用 LangSmith Deployment 做生产（`langgraph dev` 或 LangSmith Deployments）
3. **添加 Skills** - 为自己的领域能力创建 SKILL.md
4. **定制记忆** - 用命名空间作用域区分按用户 vs 共享记忆
5. **扩展** - 用专门子智能体构建多智能体系统

<br>
<br>
---
<br>

**祝构建愉快！**
